# 04 — Depstor output binaries (interactive overlay)

Stacks the 7 depstor binary rasters that **directly feed a PRMS ratio CSV** (`carea_max`, `smidx_coef`, `sro_to_dprst_perv/imperv`, `hru_percent_imperv`, `dprst_frac`) on an interactive OpenStreetMap basemap. Each layer is toggleable via the layer control (top right); colors are baked at 55% alpha so stacks of layers stay readable.

Excluded: `land_mask`, `wbody_regions`, `vpu_id`, and the intermediates that don't appear as a numerator or denominator of any output ratio (`stream_buffer`, `wbody_binary`, `onstream_binary`, `drains_to_dprst`).

> **Run this in JupyterHub on a compute node with enough `--mem`.** Folium
> reprojects each depstor raster to EPSG:4326 once and embeds an RGBA PNG into
> the notebook HTML; for CONUS `gfv2` (~361k HRUs) start with a higher `--mem`
> session. Pick the fabric via the `FABRIC` env var (or edit the first code cell).

In [ ]:
import os
from pathlib import Path

import geopandas as gpd

from gfv2_params import viz
from gfv2_params.config import load_base_config

# Fabric is selected by the FABRIC env var (set it in your JupyterHub session),
# or edit the default here.
FABRIC = os.environ.get("FABRIC", "oregon")
viz.FABRIC = FABRIC

cfg = load_base_config(fabric=FABRIC)
DATA_ROOT = Path(cfg["data_root"])
HRU_GPKG = Path(cfg["hru_gpkg"])
HRU_LAYER = cfg["hru_layer"]
ID_FEATURE = cfg["id_feature"]

print(f"FABRIC    = {FABRIC}")
print(f"data_root = {DATA_ROOT}")

## Inventory: the 7 output-binary rasters + the ratio each feeds

In [ ]:
inventory = viz.depstor_output_binary_inventory(cfg)
for e in inventory:
    print(f"  {e.name:24} color={e.color}  feeds {e.feeds}")

## Build the interactive map

Centers on the fabric centroid in lat/lon; the HRU outline is added as a thin red GeoJson layer (also toggleable). Use the layer control (top right) to show/hide individual rasters and switch basemaps.

In [ ]:
import folium

# Fabric outline once -> lat/lon
fabric_geom = gpd.read_file(HRU_GPKG, layer=HRU_LAYER)
fabric_geom = fabric_geom[fabric_geom.geometry.notna() & ~fabric_geom.geometry.is_empty]
outline_4326 = fabric_geom.dissolve().to_crs("EPSG:4326")
minx, miny, maxx, maxy = outline_4326.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

# Basemap + reasonable starting zoom (Leaflet zoom 6 ~ multi-state)
m = folium.Map(location=center, zoom_start=7, tiles="OpenStreetMap",
               control_scale=True)
# Extra basemaps under the layer control
folium.TileLayer("CartoDB positron", name="CartoDB positron").add_to(m)
folium.TileLayer("Esri.WorldImagery", attr="Tiles \u00a9 Esri",
                 name="Esri imagery").add_to(m)

# Add each depstor overlay (each one is reprojected + decimated lazily here)
for entry in inventory:
    viz.raster_to_image_overlay(
        entry.path, name=f"{entry.name}  ({entry.color})",
        color=entry.color, alpha=0.55, target_px=1200,
    ).add_to(m)

# HRU outline as a toggleable overlay
folium.GeoJson(
    outline_4326.__geo_interface__,
    name="HRU fabric outline",
    style_function=lambda _: {"color": "#000000", "weight": 1.5,
                              "fill": False},
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m  # render inline

### Optional: save the map to a standalone HTML file

Useful for sharing in a report (the HTML carries the rasters embedded as
base64 PNGs, so it's self-contained but bigger than a PNG).

In [ ]:
# Uncomment to save:
# out = DATA_ROOT.parent / f"depstor_overlay_{FABRIC}.html"
# m.save(str(out)); print(f"wrote {out}")